<div align="center">
      <img src="https://raw.githubusercontent.com/WSU-AI-in-ME/ai-in-me-1/main/img/wsu_logo_horizontal.png" alt="Wayne State University Logo" width="320">
      <h1>Homework 4: Select and Evaluate Tree Models</h1>
    </div>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WSU-AI-in-ME/ai-in-me-1/blob/main/homework/hw04_tree_and_random_forest_selection.ipynb)

    ### ME 5995: AI in Mechanical Engineering I: Fundamentals of Manufacturing Data Science

    Wayne State University

    [Course Repository](https://github.com/WSU-AI-in-ME/ai-in-me-1)

    [Homework Index](https://github.com/WSU-AI-in-ME/ai-in-me-1/blob/main/Homework_index.ipynb)

    **Version:** Student Notebook

    **Homework:** Individual graded work

    Open in Colab and save a personal copy before editing.
    Canvas controls assignment dates and submission requirements.

## Student Information

Double-click this Markdown cell to edit.

**Student name:** TODO: Enter your name  
**WSU AccessID:** TODO: Enter your WSU AccessID

## Overview and learning objectives

    Select a decision tree and Random Forest for current-cut wear classification,
    then evaluate the frozen choices on another cutter. Your task is to distinguish
    better validation fit from credible transfer, and explain class-specific errors.
    A forest is not required to outperform a tree. No points depend on winning a
    performance competition.

    You should be able to control model capacity, implement analogous fits and
    predictions, justify validation-based selection, compare class errors, and
    critique feature importance. Use Practice 7's six activities and guided
    importance example for preparation. Additional Practice examples are optional.

    Five parts, 100 points. Supplied infrastructure and the optional challenge are
    not graded. Use the notebook only; no separate report. All code and written
    responses marked TODO are required: four short response blocks, including
    two small tables. Keep the supplied comparison tables and figures.
    Packages are supplied by Colab; internet is needed for the CSV. Local users
    install numpy, pandas, matplotlib, scikit-learn and notebook support in their
    local Python environment.

    **Protocol:** c1 fitting; c4 development validation and selection; c6 evaluation
    after selection. Keep all 11 inputs and specified settings fixed. Do not inspect
    c6 to choose candidates, refit using c4, change labels, search extra settings or
    revise choices after c6. c6 has been used earlier in the course; this is a
    controlled teaching exercise, not a new untouched benchmark test.
    One development cutter and one evaluation cutter are limited evidence.
    These row groups represent cutters, not different machines/materials.
    This fixed cutter split does not perform cross-validation.

## Dataset and engineering context

    One PHM row represents one machining cut. The course feature table contains
    315 cuts per cutter c1, c4 and c6 (945 rows). Original recordings contain three
    force channels (N), three vibration channels (g), and one AE-RMS channel (V),
    documented at 50 kHz. These notebooks use supplied full-record summaries,
    not raw waveforms. Mean and SD retain their channel units.

    Target `wear_level` is a **course-defined** current-cut category from mean
    flute wear: 0 Slight-wear (<=75 micrometers), 1 Intermediate-wear
    (>75 and <150 micrometers), 2 Severe-wear (>=150 micrometers).
    These are not official challenge classes or industrial replacement limits.
    This is current-cut classification, not forecasting.

    Eleven documented sensor inputs are fixed. Wear measurements, target labels,
    cutter IDs and cut numbers are excluded from predictors. IDs define groups.
    Axis-aligned trees use threshold splits; scaling is not required here.
    This differs from distance-based methods and regularized linear models.

    Source and curation/redistribution documentation:
    [PHM official challenge](https://phmsociety.org/phm_competition/2010-phm-society-conference-data-challenge/),
    [course provenance](https://github.com/WSU-AI-in-ME/ai-in-me-1/blob/main/data/phm2010/README.md),
    [feature definitions](https://github.com/WSU-AI-in-ME/ai-in-me-1/blob/main/data/phm2010/features/DATA_DICTIONARY.md).
    Use the existing course distribution; this draft creates no new data subset.
    Cite the original challenge and course feature documentation in reused work.

In [ ]:
# Run this setup once after restarting the runtime. Imports make library names available.
# numpy handles numerical arrays; pandas handles tables; matplotlib creates figures.
# sklearn supplies the models and metrics. display shows readable notebook tables.

# PROVIDED: run once before any module. No additional installation is needed in Colab.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, confusion_matrix, ConfusionMatrixDisplay, mean_squared_error
from sklearn.inspection import permutation_importance
from sklearn.dummy import DummyClassifier

In [ ]:
# PROVIDED SETUP: one CSV row is one machining cut, not one raw signal sample.
# Force features are in N, vibration features in g, and AE-RMS features in V.
# Mean and standard deviation retain the original channel units.
# c1 supplies training rows; c4 supplies validation rows for a cutter absent from fitting.
# Keep cutter IDs and wear measurements out of the predictor columns.

# Load the documented course CSV from its public URL; the table includes inputs and metadata.
data = pd.read_csv('https://raw.githubusercontent.com/WSU-AI-in-ME/ai-in-me-1/main/data/phm2010/features/phm2010_features.csv')
# Only these named sensor summaries enter the models. Column order stays identical across datasets.
features = ['force_x_mean', 'force_x_sd', 'force_y_mean', 'force_y_sd', 'force_z_mean', 'force_z_sd', 'vibration_x_sd', 'vibration_y_sd', 'vibration_z_sd', 'ae_rms_mean', 'ae_rms_sd']
# This Boolean mask is True for c1 rows and False for all other cutter rows.
fit_rows = data['cutter_id'] == 'c1'
# This separate mask selects c4; it must not be used to train the candidate models.
valid_rows = data['cutter_id'] == 'c4'
# X contains predictor columns: c1 rows by 11 sensor features.
X_fit = data.loc[fit_rows, features]
# y contains one course-defined wear class (0, 1 or 2) per c1 row.
y_fit = data.loc[fit_rows, 'wear_level']
# Use exactly the same feature columns and order for the held-out c4 inputs.
X_valid = data.loc[valid_rows, features]
# c4 labels are used for validation scoring and selection, not fitting.
y_valid = data.loc[valid_rows, 'wear_level']
display(pd.crosstab(data.loc[fit_rows | valid_rows, 'cutter_id'], data.loc[fit_rows | valid_rows, 'wear_level']))
majority_baseline = DummyClassifier(strategy='most_frequent')
# fit learns model parameters or tree rules from the supplied input rows and matching targets.
majority_baseline.fit(X_fit, y_fit)
# predict applies the already fitted model; it does not update its learned rules.
majority_baseline_fit_prediction = majority_baseline.predict(X_fit)

In [ ]:
majority_baseline_valid_prediction = majority_baseline.predict(X_valid)
# Give true labels first and predictions second. F1 is dimensionless, with larger values better.
# macro averages class-wise F1 equally; labels fixes class order and zero_division handles undefined ratios.
majority_baseline_fit_f1 = f1_score(y_fit, majority_baseline_fit_prediction, labels=[0, 1, 2], average='macro', zero_division=0)
majority_baseline_valid_f1 = f1_score(y_valid, majority_baseline_valid_prediction, labels=[0, 1, 2], average='macro', zero_division=0)
print('Training-majority baseline')
print('Training macro-F1:', round(majority_baseline_fit_f1, 3))
print('Validation macro-F1:', round(majority_baseline_valid_f1, 3))

## Part 1: Evaluation design (15 points)

    In two sentences, explain the roles of c1 and c4 and why a shuffled pooled
    row split does not evaluate the same deployment to an absent cutter.

**Required design explanation:**

**Your response:** TODO: Write your answer here.

## Part 2: Fit a controlled candidate set (25 points)

    Complete the supplied analogous examples. Tree candidates have max_depth
    1 (completed), 2 and 4; all use min_samples_leaf=5 and random_state=42.
    Forest candidates use max_features='sqrt' (completed), 0.5 and 1.0; all have
    100 trees, max_depth=8, min_samples_leaf=5 and random_state=42.
    With 11 inputs, these consider 3, 5 and 11 features per split respectively.

    `fit(X_fit, y_fit)` learns from c1 only. `predict(X_valid)` returns a class
    for each c4 row. Supply two statements for each of four unfinished candidates.
    Constructors and scoring are provided. Adapt Practice 7 examples to write your own statements.
    The initial notebook runs safely and reports unfinished work.

In [ ]:
# GUIDED EXAMPLES: these completed fit/predict pairs show the workflow to adapt.
# Constructing a model specifies its settings; fit learns split rules from c1.
# predict uses those learned rules on c4 without learning from c4 labels.

# max_depth limits splits along a path; None removes this explicit depth bound.
# min_samples_leaf sets the minimum training sample count in every terminal leaf.
tree_1 = DecisionTreeClassifier(max_depth=1, min_samples_leaf=5, random_state=42)
# fit learns model parameters or tree rules from the supplied input rows and matching targets.
tree_1.fit(X_fit, y_fit)
# predict applies the already fitted model; it does not update its learned rules.
tree_1_valid = tree_1.predict(X_valid)
# The forest combines bootstrapped trees and random feature candidates at each split.
# n_estimators counts member trees; random_state makes the randomized fit reproducible.
# n_jobs=1 uses one worker for reproducible, modest resource use in this exercise.
forest_sqrt = RandomForestClassifier(n_estimators=100, max_depth=8,
    min_samples_leaf=5, max_features='sqrt', random_state=42, n_jobs=1)
forest_sqrt.fit(X_fit, y_fit)
forest_sqrt_valid = forest_sqrt.predict(X_valid)
tree_1_fit_prediction = tree_1.predict(X_fit)
tree_1_valid_prediction = tree_1.predict(X_valid)
# Give true labels first and predictions second. F1 is dimensionless, with larger values better.
# macro averages class-wise F1 equally; labels fixes class order and zero_division handles undefined ratios.
tree_1_fit_f1 = f1_score(y_fit, tree_1_fit_prediction, labels=[0, 1, 2], average='macro', zero_division=0)
tree_1_valid_f1 = f1_score(y_valid, tree_1_valid_prediction, labels=[0, 1, 2], average='macro', zero_division=0)
print('Tree depth 1')
print('Training macro-F1:', round(tree_1_fit_f1, 3))
print('Validation macro-F1:', round(tree_1_valid_f1, 3))
forest_sqrt_fit_prediction = forest_sqrt.predict(X_fit)

In [ ]:
forest_sqrt_valid_prediction = forest_sqrt.predict(X_valid)
forest_sqrt_fit_f1 = f1_score(y_fit, forest_sqrt_fit_prediction, labels=[0, 1, 2], average='macro', zero_division=0)
forest_sqrt_valid_f1 = f1_score(y_valid, forest_sqrt_valid_prediction, labels=[0, 1, 2], average='macro', zero_division=0)
print('Forest sqrt')
print('Training macro-F1:', round(forest_sqrt_fit_f1, 3))
print('Validation macro-F1:', round(forest_sqrt_valid_f1, 3))

In [ ]:
# REQUIRED: adapt the completed reference fit/predict pair to this candidate.
# The constructor below creates an unfitted model with the required settings.
# Training must use the c1 input table and its corresponding class labels.
# Your validation expression must predict one class for each c4 input row.
# None is a safe unfinished-work marker; replace it at the specified TODO.

# tree_2 is an unfitted model; its validation prediction must use c4.
# max_depth limits splits along a path; None removes this explicit depth bound.
# min_samples_leaf sets the minimum training sample count in every terminal leaf.
tree_2 = DecisionTreeClassifier(max_depth=2, min_samples_leaf=5, random_state=42)
# TODO: Write the training statement here.
# TODO: Replace None with a prediction expression.
tree_2_valid = None

In [ ]:
# REQUIRED: adapt the completed reference fit/predict pair to this candidate.
# The constructor below creates an unfitted model with the required settings.
# Training must use the c1 input table and its corresponding class labels.
# Your validation expression must predict one class for each c4 input row.
# None is a safe unfinished-work marker; replace it at the specified TODO.

# tree_4 is an unfitted model; its validation prediction must use c4.
# max_depth limits splits along a path; None removes this explicit depth bound.
# min_samples_leaf sets the minimum training sample count in every terminal leaf.
tree_4 = DecisionTreeClassifier(max_depth=4, min_samples_leaf=5, random_state=42)
# TODO: Write the training statement here.
# TODO: Replace None with a prediction expression.
tree_4_valid = None

In [ ]:
# REQUIRED: adapt the completed reference fit/predict pair to this candidate.
# The constructor below creates an unfitted model with the required settings.
# Training must use the c1 input table and its corresponding class labels.
# Your validation expression must predict one class for each c4 input row.
# None is a safe unfinished-work marker; replace it at the specified TODO.

# forest_half is an unfitted model; its validation prediction must use c4.
# min_samples_leaf sets the minimum training sample count in every terminal leaf.
# The forest combines bootstrapped trees and random feature candidates at each split.
# n_estimators counts member trees; random_state makes the randomized fit reproducible.
# n_jobs=1 uses one worker for reproducible, modest resource use in this exercise.
forest_half = RandomForestClassifier(n_estimators=100, max_depth=8, min_samples_leaf=5, max_features=0.5, random_state=42, n_jobs=1)
# TODO: Write the training statement here.
# TODO: Replace None with a prediction expression.
forest_half_valid = None

In [ ]:
# REQUIRED: adapt the completed reference fit/predict pair to this candidate.
# The constructor below creates an unfitted model with the required settings.
# Training must use the c1 input table and its corresponding class labels.
# Your validation expression must predict one class for each c4 input row.
# None is a safe unfinished-work marker; replace it at the specified TODO.

# forest_all is an unfitted model; its validation prediction must use c4.
# min_samples_leaf sets the minimum training sample count in every terminal leaf.
# The forest combines bootstrapped trees and random feature candidates at each split.
# n_estimators counts member trees; random_state makes the randomized fit reproducible.
# n_jobs=1 uses one worker for reproducible, modest resource use in this exercise.
forest_all = RandomForestClassifier(n_estimators=100, max_depth=8, min_samples_leaf=5, max_features=1.0, random_state=42, n_jobs=1)
# TODO: Write the training statement here.
# TODO: Replace None with a prediction expression.
forest_all_valid = None

In [ ]:
# PROVIDED REPORTING: run after completing all four candidate fit/predict pairs.
# Each model name, fitted object and prediction array must refer to the same candidate.
# Each candidate has explicit prediction and score variables; table assembly is provided.

# PROVIDED REPORTING: scores for the six fixed candidates.
# None marks an unfinished prediction; this check prevents scoring absent student outputs.
completed = (tree_2_valid is not None and tree_4_valid is not None
             and forest_half_valid is not None and forest_all_valid is not None)

In [ ]:
if completed:
    # Score the validation predictions from the candidate cell above.
    # predict applies the already fitted model; it does not update its learned rules.
    tree_1_fit_prediction = tree_1.predict(X_fit)
    # Give true labels first and predictions second. F1 is dimensionless, with larger values better.
    # macro averages class-wise F1 equally; labels fixes class order and zero_division handles undefined ratios.
    tree_1_fit_f1 = f1_score(y_fit, tree_1_fit_prediction, labels=[0, 1, 2], average='macro', zero_division=0)
    tree_1_valid_f1 = f1_score(y_valid, tree_1_valid, labels=[0, 1, 2], average='macro', zero_division=0)
    tree_1_score_row = ['tree_1', tree_1_fit_f1, tree_1_valid_f1]
    # Score the validation predictions from the candidate cell above.
    tree_2_fit_prediction = tree_2.predict(X_fit)
    tree_2_fit_f1 = f1_score(y_fit, tree_2_fit_prediction, labels=[0, 1, 2], average='macro', zero_division=0)
    tree_2_valid_f1 = f1_score(y_valid, tree_2_valid, labels=[0, 1, 2], average='macro', zero_division=0)
    tree_2_score_row = ['tree_2', tree_2_fit_f1, tree_2_valid_f1]
    # Score the validation predictions from the candidate cell above.
    tree_4_fit_prediction = tree_4.predict(X_fit)
    tree_4_fit_f1 = f1_score(y_fit, tree_4_fit_prediction, labels=[0, 1, 2], average='macro', zero_division=0)
    tree_4_valid_f1 = f1_score(y_valid, tree_4_valid, labels=[0, 1, 2], average='macro', zero_division=0)
    tree_4_score_row = ['tree_4', tree_4_fit_f1, tree_4_valid_f1]
    # Score the validation predictions from the candidate cell above.
    forest_sqrt_fit_prediction = forest_sqrt.predict(X_fit)
    forest_sqrt_fit_f1 = f1_score(y_fit, forest_sqrt_fit_prediction, labels=[0, 1, 2], average='macro', zero_division=0)
    forest_sqrt_valid_f1 = f1_score(y_valid, forest_sqrt_valid, labels=[0, 1, 2], average='macro', zero_division=0)

In [ ]:
if completed:
    forest_sqrt_score_row = ['forest_sqrt', forest_sqrt_fit_f1, forest_sqrt_valid_f1]
    # Score the validation predictions from the candidate cell above.
    forest_half_fit_prediction = forest_half.predict(X_fit)
    forest_half_fit_f1 = f1_score(y_fit, forest_half_fit_prediction, labels=[0, 1, 2], average='macro', zero_division=0)
    forest_half_valid_f1 = f1_score(y_valid, forest_half_valid, labels=[0, 1, 2], average='macro', zero_division=0)
    forest_half_score_row = ['forest_half', forest_half_fit_f1, forest_half_valid_f1]
    # Score the validation predictions from the candidate cell above.
    forest_all_fit_prediction = forest_all.predict(X_fit)
    forest_all_fit_f1 = f1_score(y_fit, forest_all_fit_prediction, labels=[0, 1, 2], average='macro', zero_division=0)
    forest_all_valid_f1 = f1_score(y_valid, forest_all_valid, labels=[0, 1, 2], average='macro', zero_division=0)
    forest_all_score_row = ['forest_all', forest_all_fit_f1, forest_all_valid_f1]
    candidate_names = ['tree_1', 'tree_2', 'tree_4', 'forest_sqrt', 'forest_half', 'forest_all']
    candidate_models = [tree_1, tree_2, tree_4, forest_sqrt, forest_half, forest_all]
    # Collect one reporting row per candidate. Each row keeps its model name and two scores together.
    score_rows = [tree_1_score_row, tree_2_score_row, tree_4_score_row,
                  forest_sqrt_score_row, forest_half_score_row, forest_all_score_row]
    candidate_table = pd.DataFrame(score_rows, columns=['Model', 'Training macro-F1', 'Validation macro-F1'])
    candidate_table['Training minus validation'] = candidate_table['Training macro-F1'] - candidate_table['Validation macro-F1']
    candidate_table['Actual tree depth'] = [tree_1.get_depth(), tree_2.get_depth(), tree_4.get_depth(), None, None, None]
    candidate_table['Tree leaves'] = [tree_1.get_n_leaves(), tree_2.get_n_leaves(), tree_4.get_n_leaves(), None, None, None]
    # Round only the display; retain full-precision values when comparing candidates or checking ties.
    display(candidate_table.round(3))
else:
    print('Incomplete Part 2: finish all four fit/predict pairs before selection.')

In [ ]:
# PROVIDED: these unrounded validation scores support selection and exact-tie checks.
# Run unchanged after completing all candidates; the table above is rounded for readability.
if completed:
    print('tree_1:', tree_1_valid_f1)
    print('tree_2:', tree_2_valid_f1)
    print('tree_4:', tree_4_valid_f1)
    print('forest_sqrt:', forest_sqrt_valid_f1)
    print('forest_half:', forest_half_valid_f1)
    print('forest_all:', forest_all_valid_f1)

## Part 3: Select and freeze two models (25 points)

    Choose one tree and one forest using the largest c4 validation macro-F1 in
    each family, **before viewing c6**. Use the unrounded validation scores printed
    above to compare candidates and check exact ties. You may record three decimal
    places in your response table.
    For an exact tie, choose the smaller tree depth or, for forests, the earlier
    candidate in the listed order. Supply the two model objects in the cell below.

    Fill the two-row selection table, then answer the two short questions below.
    Use the displayed actual depth and leaf count, not equal scores alone, when
    discussing structure. Freeze your choices before running Part 4.

In [ ]:
# Use the c4 validation table to choose one fitted candidate from each model family.
# Assign the fitted model object, not a string name or a newly constructed model.
# Choose using the stated validation rule; the supplied code only checks for unfinished work.
# Keep these choices fixed after observing c6. Do not refit on c4.

selected_tree = None  # TODO: Replace None with your chosen fitted tree object.
selected_forest = None  # TODO: Replace None with your chosen fitted forest object.

# PROVIDED: skip later reporting until both choices are filled in.
selection_ready = False
if completed and selected_tree is not None and selected_forest is not None:
    selection_ready = True
    print('Both models selected. Keep your choices fixed for Part 4.')
else:
    print('Incomplete Part 3: finish candidate fits and select both models.')

**Required selection record**

    | Family | Selected model | c4 validation macro-F1 |
    |---|---|---|
    | Tree | TODO | TODO |
    | Forest | TODO | TODO |

    1. Why can depth limits 2 and 4 give the same actual depth? Use the structure columns. One sentence.

    TODO: Answer here.

    2. Why is the selected c4 score not an independent final-test score? One sentence.

    TODO: Answer here.

## Part 4: Evaluate frozen choices and diagnose errors (25 points)

    Run the supplied evaluation only after Part 3. Models remain fitted to c1;
    there is no refitting on c4. Evaluation is based on c6, not the training table.
    Rerun only for debugging or reproducibility; do not revise choices from c6.

    Read the supplied score/error table and two matrices. Fill only the two-row
    error record and answer three short questions; do not copy the full output
    tables into your response. Course wear classes do not establish safe operation.

In [ ]:
# PROVIDED EVALUATION: run only after choosing and freezing both fitted models.
# c6 is absent from fitting and selection in this homework, but was used earlier in the course.
# Prediction does not refit the models. Keep c6 results out of later model revisions.
# In each matrix, rows are actual classes and columns are predicted classes.
# Class 2 means course-defined Severe-wear, not an industrial safety threshold.

# PROVIDED: c6 is accessed for model evaluation only after selection succeeds.
if selection_ready:
    # c6 is selected only after the fitting and validation-selection stages are complete.
    evaluation_rows = data['cutter_id'] == 'c6'
    # These are c6 sensor inputs in the same order used to fit the selected models.
    X_evaluation = data.loc[evaluation_rows, features]
    # Use c6 labels to evaluate frozen predictions; do not train or choose settings with them.
    y_evaluation = data.loc[evaluation_rows, 'wear_level']
    # predict applies the already fitted model; it does not update its learned rules.
    tree_evaluation = selected_tree.predict(X_evaluation)
    forest_evaluation = selected_forest.predict(X_evaluation)
    # The first index is the true class; the second index is the predicted class.
    tree_matrix = confusion_matrix(y_evaluation, tree_evaluation, labels=[0,1,2])
    # Explicit labels keep the row/column order fixed even if a prediction omits a class.
    forest_matrix = confusion_matrix(y_evaluation, forest_evaluation, labels=[0,1,2])
    display(pd.crosstab(data.loc[evaluation_rows, 'cutter_id'], y_evaluation))
    # Give true labels first and predictions second. F1 is dimensionless, with larger values better.
    # macro averages class-wise F1 equally; labels fixes class order and zero_division handles undefined ratios.
    comparison_validation_f1 = f1_score(y_evaluation, tree_evaluation, labels=[0, 1, 2], average='macro', zero_division=0)
    comparison_validation_f1_2 = f1_score(y_evaluation, forest_evaluation, labels=[0, 1, 2], average='macro', zero_division=0)
    evaluation_table = pd.DataFrame([['Tree', comparison_validation_f1, tree_matrix[2, 0] + tree_matrix[2, 1], tree_matrix[0, 2] + tree_matrix[1, 2]], ['Forest', comparison_validation_f1_2, forest_matrix[2, 0] + forest_matrix[2, 1], forest_matrix[0, 2] + forest_matrix[1, 2]]], columns=['Model', 'c6 macro-F1', 'Missed Severe', 'False Severe'])
    # Round only the display; retain full-precision values when comparing candidates or checking ties.
    display(evaluation_table.round(3))
    majority_baseline_query_prediction = majority_baseline.predict(X_evaluation)

In [ ]:
if selection_ready:
    comparison_validation_f1 = f1_score(y_evaluation, majority_baseline_query_prediction, labels=[0, 1, 2], average='macro', zero_division=0)
    print('c6 training-majority baseline macro-F1:', round(comparison_validation_f1, 3))
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
    ConfusionMatrixDisplay(tree_matrix, display_labels=[0,1,2]).plot(ax=axes[0], cmap='Blues', colorbar=False)
    ConfusionMatrixDisplay(forest_matrix, display_labels=[0,1,2]).plot(ax=axes[1], cmap='Blues', colorbar=False)
    axes[0].set_title('c6 evaluation: selected tree')
    axes[1].set_title('c6 evaluation: selected forest')
    plt.tight_layout()
    plt.show()
    print('Class key: 0 Slight-wear; 1 Intermediate-wear; 2 Severe-wear')
else:
    print('Evaluation deferred until Parts 2 and 3 are complete.')

**Required error record**

    | Model | Largest off-diagonal error: actual class to predicted class | Count |
    |---|---|---|
    | Tree | TODO | TODO |
    | Forest | TODO | TODO |

    Answer each question in one sentence using the displayed outputs.

    1. Does the c4 model ranking persist on c6, and do both models beat the majority baseline?

    TODO: Answer here.

    2. Which error trade-off matters if missed Severe is costly but false Severe also triggers unnecessary intervention?

    TODO: Answer here.

    3. State one limitation on generalizing this result.

    TODO: Answer here.

## Part 5: Critique sensor importance (10 points)

    Use the supplied selected-forest impurity importance and **c4 validation**
    permutation importance. c6 is not used to select features or model settings.
    Name the top feature under each definition, then give one sentence explaining
    why these rankings do not prove physical causality. Correlated inputs are a
    useful consideration. Repeat SD is explained in the supplied comments and is
    not an additional written task.
    Do not change the feature set after inspecting importance.

In [ ]:
# PROVIDED INTERPRETATION: examine the selected forest without changing its inputs.
# Impurity importance summarizes training splits; permutation importance uses c4.
# A shuffle breaks one column's row-wise correspondence while other columns stay fixed.
# Repeat SD describes variation across shuffles of these same data, not new cutters.

if selection_ready:
    # Refitting is not performed: repeatedly shuffle one validation column and measure the score drop.
    importance = permutation_importance(selected_forest, X_valid, y_valid,
        # Five shuffles give a mean and repeat SD for each column; the score is macro-F1 here.
        scoring='f1_macro', n_repeats=5, random_state=42, n_jobs=1)
    # Place both importance definitions side by side; their numerical scales and meanings differ.
    importance_table = pd.DataFrame()
    importance_table['Feature'] = features
    importance_table['Training impurity importance'] = selected_forest.feature_importances_
    importance_table['c4 permutation mean drop'] = importance.importances_mean
    importance_table['Repeat SD'] = importance.importances_std
    sorted_importance = importance_table.sort_values('c4 permutation mean drop', ascending=False)
    # Round only the display; retain full-precision values when comparing candidates or checking ties.
    rounded_importance = sorted_importance.round(3)
    display(rounded_importance)
else:
    print('Importance deferred until model selection is complete.')

**Required importance record**

**Top impurity feature:** TODO

**Top c4 permutation feature:** TODO

**Causal limitation (one sentence):** TODO

## Optional challenge: plan a stronger study

    Propose a nested cutter-grouped selection/evaluation design for a future dataset
    containing many cutters. Explain which groups enter inner selection and outer
    evaluation. No extra model runs are required and this is not graded.

## Submission checklist

    Rendered notebook checkboxes are not clickable. To record completion, edit
    this Markdown cell and change `[ ]` to `[x]`.

    - [ ] Enter name and WSU AccessID.
    - [ ] Complete four fit/predict pairs and two model selections.
    - [ ] Complete four short response blocks, including two small tables; no essays.
    - [ ] Preserve candidate scores, frozen choices, evaluation table, matrices and importance.
    - [ ] Restart the runtime and Run all; resolve errors and unfinished messages.
    - [ ] Submit `HW04_Firstname_Lastname.ipynb` and `HW04_Firstname_Lastname.pdf` through Canvas.
    - [ ] Keep required outputs readable; no separate report is needed.

    PDF export issues receive corrective feedback rather than point deductions;
    the notebook is used to assess work obscured by export. Both files are required.
    Canvas controls the due date. Guided code and the optional challenge are excluded
    from assessment.

## References

    - [scikit-learn: Decision trees](https://scikit-learn.org/stable/modules/tree.html)
    - [scikit-learn: Ensembles](https://scikit-learn.org/stable/modules/ensemble.html)
    - [scikit-learn: Permutation importance](https://scikit-learn.org/stable/modules/permutation_importance.html)
    - [PHM Society: 2010 Data Challenge](https://phmsociety.org/phm_competition/2010-phm-society-conference-data-challenge/)
    - [Course PHM feature documentation](https://github.com/WSU-AI-in-ME/ai-in-me-1/blob/main/data/phm2010/features/README.md)